# 06 — Back out the surface net shortwave from the penetrating `oceQsw`

Notebook 05 established that `mit/oceQsw` carries ~12% of the surface net shortwave.
The data descriptor points to the responsible MITgcm code, which makes the stream
**exactly invertible**:

- `model/src/swfrac.F`: the Paulson & Simpson (1977) two-band profile
  $f(z) = R\,e^{-z/a_1} + (1{-}R)\,e^{-z/a_2}$ with hard-coded Jerlov IA
  coefficients ($R{=}0.62$, $a_1{=}0.6$ m, $a_2{=}20$ m), zeroed below 200 m —
  a function of depth **only** (no solar zenith angle, no clouds).
- `model/src/ini_forcing.F` evaluates $f$ once at the level interfaces
  (`SWFracK(k) = rF(k) - rF(1)`), and `model/src/apply_forcing.F` deposits
  $Q_{sw}\,[f(z_k) - f(z_{k+1})]$ in layer $k$.

Below ~3 m the $a_1$ band is extinct, so the flux at any fixed interface $z^*$ is a
**single constant** $c = f(z^*)$ times the surface flux. Hence, if `oceQsw` is the
penetrating flux at a fixed interface,

$$ SW_{surf} = \mathtt{oceQsw} / c \quad\text{(exact)}, $$

and the same $f$ then gives the full 3-D shortwave deposition. This notebook

1. determines $c$ empirically — a zero-intercept regression of the binned July-2020
   mean `oceQsw` (positive down) against the GEOS surface net SW from notebook 04;
2. identifies the interface: matches $c$ against $f$ evaluated at the run's own
   vertical grid (`mit/grid/DRF.data`) — adjacent interfaces differ by >10% in
   fraction, so the identification is unambiguous;
3. reconstructs $SW_{surf}$ and quantifies how exact the inversion is.

**Prerequisites**: run notebook 04 first (its cache `sw_geos_2020-07_1deg.nc` supplies
the GEOS mean and the ocean mask). If the notebook-04-v1 cache
`qsw_model_2020-07_1deg.nc` (the binned July mean of raw `oceQsw`) was deleted, the
cell below recomputes it (~60 GB of reads).

In [ ]:
# Environment check: run on SciServer (Kraken domain, with the Poseidon DYAMOND
# ceph volume attached), or set DYAMOND_ROOT to a local subset.
# SciServer containers do not persist `pip install --user` across restarts, so
# fall back to importing directly from the repo's src/ tree if needed.
try:
    from dyamond_fluxes import dyamond_root
except ModuleNotFoundError:
    import sys
    from pathlib import Path as _P

    sys.path.insert(0, str((_P.cwd() / ".." / "src").resolve()))
    from dyamond_fluxes import dyamond_root

root = dyamond_root()
print(f"DYAMOND root: {root}")

In [ ]:
from pathlib import Path

MONTH_START, MONTH_END = "2020-07-01", "2020-08-01"
SUBSAMPLE = 3
DLON = DLAT = 1.0
GEOS_CACHE = Path(f"sw_geos_{MONTH_START[:7]}_1deg.nc")    # from notebook 04 (v2)
QSW_CACHE = Path(f"qsw_model_{MONTH_START[:7]}_1deg.nc")   # from notebook 04 (v1)
FIGDIR = Path("../figures")
FIGDIR.mkdir(exist_ok=True)

## Load the two July-2020 means (recompute `oceQsw` if its cache is gone)

In [ ]:
import numpy as np
import xarray as xr

if not GEOS_CACHE.exists():
    raise FileNotFoundError(f"{GEOS_CACHE} not found - run notebook 04 first")
geos_ds = xr.open_dataset(GEOS_CACHE)
sw_geos, ocean_frac = geos_ds["sw_model"], geos_ds["ocean_frac"]

if QSW_CACHE.exists():
    qsw_down = xr.open_dataarray(QSW_CACHE)
    # The v1 cache stored the RAW (positive-up) mean; newer runs store positive-down.
    if float(qsw_down.where(abs(qsw_down.lat) < 30).mean()) < 0:
        qsw_down = -qsw_down
        print("cache held positive-up values; negated to positive-down")
else:
    print(f"{QSW_CACHE} missing - recomputing the July oceQsw mean (~60 GB of reads)")
    from dask.distributed import Client, LocalCluster

    from dyamond_fluxes import bin_to_latlon, open_ocean_dataset, to_positive_down

    client = Client(LocalCluster(n_workers=4, threads_per_worker=2, memory_limit="8GB"))
    ds = open_ocean_dataset(["oceQsw"])
    q = ds["oceQsw"].sel(time=slice(MONTH_START, MONTH_END))
    q = q.isel(time=slice(None, None, SUBSAMPLE))
    qm = to_positive_down(q).mean("time").where(ds["Depth"] > 0).load()
    qsw_down = bin_to_latlon(qm, ds["XC"], ds["YC"], area=ds["rA"], dlon=DLON, dlat=DLAT)
    qsw_down.name = "qsw_model"
    qsw_down.to_netcdf(QSW_CACHE)

print(f"oceQsw (down) ocean mean: "
      f"{float(qsw_down.where(ocean_frac > 0.5).mean()):6.1f} W m-2")
print(f"GEOS SW      ocean mean: "
      f"{float(sw_geos.where(ocean_frac > 0.5).mean()):6.1f} W m-2")

## 1. The constant $c$: zero-intercept regression over open-ocean bins

$c = \sum w\,x\,y / \sum w\,x^2$ with $x$ = GEOS surface SW, $y$ = `oceQsw` (down),
$w = \cos\varphi$, over ocean bins within 60°S–60°N. If the penetrating-flux model is
right, the scatter collapses onto a single line through the origin.

In [ ]:
import matplotlib.pyplot as plt

ok = (
    (ocean_frac > 0.5)
    & (abs(sw_geos.lat) < 60)
    & sw_geos.notnull()
    & qsw_down.notnull()
)
w = np.cos(np.deg2rad(sw_geos.lat)).broadcast_like(sw_geos).where(ok, 0.0)
x, y = sw_geos.where(ok), qsw_down.where(ok)

c = float((w * x * y).sum() / (w * x * x).sum())
resid = y - c * x
r2 = 1.0 - float((w * resid**2).sum() / (w * (y - float(y.weighted(w).mean())) ** 2).sum())
rel_rms = float(np.sqrt((w * resid**2).sum() / (w * y**2).sum()))
print(f"c        = {c:.5f}")
print(f"R^2      = {r2:.5f}")
print(f"relative RMS residual = {rel_rms:.2%}")

fig, ax = plt.subplots(figsize=(6, 6))
ax.hexbin(x.values.ravel(), y.values.ravel(), gridsize=60, mincnt=1, cmap="viridis")
xx = np.array([0.0, float(x.max())])
ax.plot(xx, c * xx, "r", lw=1.5, label=f"y = {c:.4f} x")
ax.set_xlabel("GEOS surface net SW (W m$^{-2}$)")
ax.set_ylabel("oceQsw, positive down (W m$^{-2}$)")
ax.set_title("July-2020 means, 1° ocean bins (60°S–60°N)")
ax.legend()
fig.savefig(FIGDIR / "qsw_backout_regression.png", dpi=200, bbox_inches="tight")

## 2. Which interface is it? Match $c$ against the run's vertical grid

`mit/grid/DRF.data` holds the 90 layer thicknesses; interfaces sit at their cumulative
sums (ini_forcing.F: `SWFracK(k) = rF(k) - rF(1)`).

In [ ]:
from dyamond_fluxes import infer_reference_depth, interface_fractions, sw_fraction
from dyamond_fluxes.mds import mit_dir

raw = (mit_dir() / "grid" / "DRF.data").read_bytes()
drf = next(
    a for dt in (">f4", ">f8")
    if (a := np.frombuffer(raw, dtype=dt)).size <= 200
    and np.all(np.isfinite(a)) and np.all(a > 0) and a[0] < 10
)
print(f"{drf.size} layers; top thicknesses: {np.round(drf[:8].astype(float), 2)}")

z_int, frac_int = interface_fractions(drf)
z_star = infer_reference_depth(c)
k_near = int(np.argmin(abs(frac_int - c)))
print(f"\nfree-depth inversion:  f(z)=c at z* = {z_star:.2f} m")
print(f"nearest interface:     k={k_near} (base of layer {k_near}), "
      f"z={z_int[k_near]:.2f} m, f={frac_int[k_near]:.5f}")

print("\n  k   z_int [m]   f(z)      (top of the column)")
for k in range(min(16, z_int.size)):
    marker = "  <-- match" if k == k_near else ""
    print(f"  {k:2d}   {z_int[k]:7.2f}   {frac_int[k]:.5f}{marker}")

## 3. Reconstruct the surface net SW and check it

If the regression above is tight (relative RMS of a few percent), the reconstruction
$SW_{surf} = \mathtt{oceQsw}/c$ is a valid model field in its own right — an
*ocean-grid, ocean-timebase* surface SW that also makes
$Q_{ns} = Q_{net} - SW_{surf}$ meaningful again. Prefer the exact grid constant
$f(z_k)$ over the regressed $c$ once the interface is identified (they should agree
to ~1%).

In [ ]:
from dyamond_fluxes import backout_surface_sw

C_USED = float(frac_int[k_near])  # exact grid constant at the identified interface
sw_rec = backout_surface_sw(qsw_down, C_USED)

fig, axes = plt.subplots(2, 1, figsize=(11, 8), sharex=True, sharey=True)
for ax, (da, label) in zip(
    axes,
    [(sw_rec.where(ok), f"oceQsw / {C_USED:.4f} (reconstructed)"),
     (sw_geos.where(ok), "GEOS net surface SW")],
):
    pc = ax.pcolormesh(da.lon, da.lat, da, cmap="inferno", vmin=0, vmax=320)
    ax.set_title(f"{label} — July 2020 mean")
fig.colorbar(pc, ax=axes, shrink=0.8, label="W m$^{-2}$")
fig.savefig(FIGDIR / "qsw_backout_maps.png", dpi=200, bbox_inches="tight")

diff = (sw_rec - sw_geos).where(ok)
print(f"reconstruction minus GEOS: mean {float(diff.weighted(w).mean()):+.1f} W m-2, "
      f"RMS {float(np.sqrt((diff**2).weighted(w).mean())):.1f} W m-2")

## 4. Bonus: the 3-D shortwave deposition profile

With $SW_{surf}$ in hand, apply_forcing.F's discretization gives the heating deposited
in each layer: $\Delta Q_k = SW_{surf}\,[f(z_k) - f(z_{k+1})]$.

In [ ]:
absorbed = -np.diff(frac_int)  # fraction absorbed per layer
fig, ax = plt.subplots(figsize=(6, 5))
zc = 0.5 * (z_int[:-1] + z_int[1:])
ax.barh(zc[:25], 100 * absorbed[:25], height=0.8 * drf[:25].astype(float))
ax.invert_yaxis()
ax.set_xlabel("% of surface net SW absorbed in layer")
ax.set_ylabel("depth (m)")
ax.set_title("Two-band SW deposition on the LLC2160 vertical grid (Jerlov IA)")
ax.grid(alpha=0.3)
fig.savefig(FIGDIR / "qsw_deposition_profile.png", dpi=200, bbox_inches="tight")
print(f"absorbed in layer 1: {100 * absorbed[0]:.1f}%; "
      f"below 200 m: {100 * frac_int[z_int <= 200][-1]:.2f}%")

## Caveats on exactness

- The inversion is exact **only if** the dumped `oceQsw` is the penetrating flux at a
  fixed interface — which the regression verifies empirically. A relative RMS residual
  beyond a few percent means part of the mismatch is real (GEOS vs ocean timebase and
  sampling, 15-min tavg vs hourly instantaneous, binning), not a broken assumption.
- `swfrac.F` hard-codes Jerlov IA; if the run had overridden the coefficients, the
  *empirical* $c$ still makes the reconstruction correct — only the depth attribution
  would shift.
- Under sea ice the coupler's SW pathway differs; keep the reconstruction (and any
  $Q_{ns}$ built from it) to the open ocean, as everywhere else in this repo.
- For CERES validation keep using the GEOS field directly (notebook 04): it is the
  same interface quantity CERES estimates, with no reconstruction step. The back-out's
  value is the ocean-side budget ($Q_{ns}$) and the 3-D absorption profile.

Report the `oceQsw` readme mismatch to the data producers — [contact/citation to be
inserted].